# Part 2 — Local Machine Learning
After **Part 1: Data Preparation**. You now have tokenized IMDB reviews split among four simulated streaming platforms. In **Part 2**, you will need to design, train, evaluate, and compare **independent local sentiment classifiers**.





## 0. Setup — from Data Preparation to Local Learning

**Theory.** A supervised model learns a function mapping numerical review features to a sentiment label. The exported token IDs represent words or subwords; they are **not** meaningful continuous values by themselves. A neural network can learn embeddings and use them to predict **negative (0)** or **positive (1)**.

An *epoch* is one pass through the local training data. A *batch* is a smaller set of samples processed in one optimizer step.



**ML concepts to be able to understand and explain:**
- **Forward pass:** tokenized input → neural network → raw logits for negative/positive.
- **Cross-entropy:** the model is penalized when it gives low probability to the correct label, e.g. $\mathcal{L}=-\log p(y_{\mathrm{true}}\mid x)$.
- **Backward pass:** `loss.backward()` computes gradients $\nabla_\theta \mathcal{L}$.
- **Optimizer step:** updates parameters; ordinary gradient descent illustrates the idea $\theta_{t+1}=\theta_t-\eta\nabla_\theta\mathcal{L}$ (AdamW uses a different update rule).
- **Validation:** measure accuracy, macro-F1 and loss on held-out **local** examples without weight updates.


In [ ]:
# Colab normally includes PyTorch. Install other libraries used below.
%pip -q install "transformers>=4.40,<6" "scikit-learn>=1.4,<2" "pandas>=2,<3" "matplotlib>=3.7,<4"


In [ ]:
import copy
import json
import random
import zipfile
from pathlib import Path
from collections import Counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
from transformers import AutoTokenizer

# ---- Choose and justify your values. These are starting points, not requirements.
SEED = 42
BASELINE_SCENARIO = "iid"
COMPARE_SCENARIO = "non_iid"
FIRST_CLIENT = 0
BATCH_SIZE = 32               # YOUR CHOICE
LEARNING_RATE = 1e-3         # YOUR CHOICE
EPOCHS = 2                   # YOUR CHOICE
EMBED_DIM = 64               # YOUR CHOICE
HIDDEN_DIM = 64              # YOUR CHOICE
DROPOUT = 0.2                # YOUR CHOICE

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RESULTS_DIR = Path("imdb_local_learning_results")
RESULTS_DIR.mkdir(exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
print("Device:", DEVICE, "| PyTorch:", torch.__version__)


Device: cpu | PyTorch: 2.14.1+cpu


### Bring your Part 1 ZIP into Colab

Upload **`imdb_fl_artifacts.zip`** (created by the previous notebook). The archive is expected to contain:

```text
imdb_fl_artifacts/
  manifest.json
  global_test.jsonl
  iid/client_0_train.jsonl ... client_3_val.jsonl
  non_iid/client_0_train.jsonl ... client_3_val.jsonl
  quantity_skew/client_0_train.jsonl ... client_3_val.jsonl
```

If the folder is already present, the upload is skipped. **Do not rerun cleaning/tokenization here**: Part 2 must use the actual output of Part 1.


In [ ]:
DATA_ROOT = Path("imdb_fl_artifacts")
ZIP_PATH = Path("imdb_fl_artifacts.zip")

if not (DATA_ROOT / "manifest.json").is_file():
    if not ZIP_PATH.is_file():
        try:
            from google.colab import files
            print("Upload imdb_fl_artifacts.zip from Part 1:")
            uploaded = files.upload()
            if ZIP_PATH.name not in uploaded:
                raise FileNotFoundError("Please upload the exact ZIP named imdb_fl_artifacts.zip")
        except ImportError as exc:
            raise FileNotFoundError("Put imdb_fl_artifacts.zip beside this notebook") from exc
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        # The ZIP comes from your own notebook;
        zf.extractall(".")

assert (DATA_ROOT / "manifest.json").is_file(), "Cannot find manifest.json after unzip"
manifest = json.loads((DATA_ROOT / "manifest.json").read_text(encoding="utf-8"))
NUM_CLIENTS = int(manifest["num_clients"])
MAX_LENGTH = int(manifest["max_length"])
TOKENIZER_NAME = manifest["tokenizer"]
AVAILABLE_SCENARIOS = list(manifest["scenarios"].keys())

print("From Part 1:", {"clients": NUM_CLIENTS, "max_length": MAX_LENGTH,
                       "tokenizer": TOKENIZER_NAME, "scenarios": AVAILABLE_SCENARIOS,
                       "fast_mode": manifest.get("fast_mode")})
assert BASELINE_SCENARIO in AVAILABLE_SCENARIOS
assert COMPARE_SCENARIO in AVAILABLE_SCENARIOS
assert 0 <= FIRST_CLIENT < NUM_CLIENTS
assert NUM_CLIENTS == 4, "This course lab expects four clients"
if manifest.get("fast_mode"):
    print("WARNING: Part 1 FAST_MODE=True — use for a smoke test only. Re-export full data for submission.")

# We only retrieve the tokenizer *metadata* for a shared vocabulary size and padding ID.You have to add code here
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
VOCAB_SIZE = tokenizer.vocab_size
PAD_TOKEN_ID = tokenizer.pad_token_id


Upload imdb_fl_artifacts.zip from Part 1:


Saving imdb_fl_artifacts.zip to imdb_fl_artifacts.zip
From Part 1: {'clients': 4, 'max_length': 256, 'tokenizer': 'distilbert-base-uncased', 'scenarios': ['iid', 'non_iid', 'quantity_skew'], 'fast_mode': False}


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]



## 1. TODO 1 — Load and audit the client data

**Theory.** Client isolation is the core of the simulation. The `iid`, `non_iid` and `quantity_skew` directories represent **alternative partitions** of the *same* IMDB training pool — not three extra datasets to concatenate. Each review record includes:

- `input_ids`: integer token IDs, length `MAX_LENGTH`;
- `attention_mask`: 1 for tokens and 0 for padding;
- `label`: 0 (negative) or 1 (positive);
- `source_id`: original row identifier, useful to check disjointness.

**Your implementation.** Finish `audit_client(...)`: load `client_{id}_train.jsonl` and `client_{id}_val.jsonl`; verify lengths, allowed labels, manifest counts and non-overlapping source IDs. Return a dictionary with **exactly** the keys in the function docstring.

**Freedom:** Add extra checks or graphs (class balance, examples, attention-mask lengths). Explain any anomalies. Do not use `global_test.jsonl` for model selection.


In [ ]:
def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open(encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def client_path(scenario, client_id, split):
    assert scenario in AVAILABLE_SCENARIOS
    assert split in ("train", "val")
    return DATA_ROOT / scenario / f"client_{client_id}_{split}.jsonl"


def audit_client(scenario, client_id):
    """Return {'train_rows': list, 'val_rows': list,
                'n_train': int, 'n_val': int,
                'train_neg_pos': [int, int], 'val_neg_pos': [int, int]}.
    Verify schema, MAX_LENGTH, source_id disjointness, and manifest counts.
    """
    # TODO 1 — YOUR CODE HERE
    # Helpful: read_jsonl(...), client_path(...), Counter(...), manifest['clients']
    train_rows = read_jsonl(client_path(scenario, client_id, "train"))
    val_rows = read_jsonl(client_path(scenario, client_id, "val"))

    for rows in (train_rows, val_rows):
        for row in rows:
            assert all(k in row for k in ("input_ids", "attention_mask", "label", "source_id"))
            assert len(row["input_ids"]) == MAX_LENGTH
            assert len(row["attention_mask"]) == MAX_LENGTH
            assert int(row["label"]) in (0, 1)
            assert set(row["attention_mask"]).issubset({0, 1})

    train_ids = [row["source_id"] for row in train_rows]
    val_ids = [row["source_id"] for row in val_rows]

    assert len(train_ids) == len(set(train_ids))
    assert len(val_ids) == len(set(val_ids))
    assert set(train_ids).isdisjoint(val_ids)

    train_neg_pos = [sum(int(row["label"]) == 0 for row in train_rows), sum(int(row["label"]) == 1 for row in train_rows)]
    val_neg_pos = [sum(int(row["label"]) == 0 for row in val_rows), sum(int(row["label"]) == 1 for row in val_rows)]

    expected = manifest["clients"][scenario][str(client_id)]

    assert len(train_rows) == expected["train"]
    assert len(val_rows) == expected["val"]
    assert train_neg_pos == expected["train_labels_neg_pos"]
    assert val_neg_pos == expected["val_labels_neg_pos"]

    return {
        "train_rows": train_rows,
        "val_rows": val_rows,
        "n_train": len(train_rows),
        "n_val": len(val_rows),
        "train_neg_pos": train_neg_pos,
        "val_neg_pos": val_neg_pos
    }
    raise NotImplementedError("TODO 1: audit_client")


In [ ]:

client0 = audit_client(BASELINE_SCENARIO, FIRST_CLIENT)
expected = manifest["clients"][BASELINE_SCENARIO][str(FIRST_CLIENT)]
assert client0["n_train"] == expected["train"] == len(client0["train_rows"])
assert client0["n_val"] == expected["val"] == len(client0["val_rows"])
assert client0["train_neg_pos"] == expected["train_labels_neg_pos"]
assert client0["val_neg_pos"] == expected["val_labels_neg_pos"]
train_ids = [r["source_id"] for r in client0["train_rows"]]
val_ids = [r["source_id"] for r in client0["val_rows"]]
assert len(set(train_ids)) == len(train_ids)
assert len(set(val_ids)) == len(val_ids)
assert set(train_ids).isdisjoint(val_ids)
for r in client0["train_rows"][:10] + client0["val_rows"][:10]:
    assert len(r["input_ids"]) == len(r["attention_mask"]) == MAX_LENGTH
    assert int(r["label"]) in (0, 1)
    assert set(r["attention_mask"]).issubset({0, 1})
print("PASS TODO 1:", client0["n_train"], "train,", client0["n_val"], "val")
print("Train [negative, positive]:", client0["train_neg_pos"])
print("Validation [negative, positive]:", client0["val_neg_pos"])


PASS TODO 1: 5000 train, 1250 val
Train [negative, positive]: [2500, 2500]
Validation [negative, positive]: [625, 625]


## 2. TODO 2 — PyTorch Dataset and DataLoader



In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, rows):
        self.input_ids = torch.tensor([r["input_ids"] for r in rows], dtype=torch.long)
        self.attention_mask = torch.tensor([r["attention_mask"] for r in rows], dtype=torch.long)
        self.labels = torch.tensor([int(r["label"]) for r in rows], dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
            "labels": self.labels[idx],
        }


def make_loaders(train_rows, val_rows, batch_size):
    train_ds = ReviewDataset(train_rows)
    val_ds = ReviewDataset(val_rows)
    g = torch.Generator()
    g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, generator=g)
    validation_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    return train_loader, validation_loader


In [ ]:
# CHECK 2 — do not edit
train_loader, val_loader = make_loaders(client0["train_rows"], client0["val_rows"], BATCH_SIZE)
batch = next(iter(train_loader))
assert set(batch) == {"input_ids", "attention_mask", "labels"}
assert batch["input_ids"].ndim == 2 and batch["input_ids"].shape[1] == MAX_LENGTH
assert batch["attention_mask"].shape == batch["input_ids"].shape
assert batch["labels"].ndim == 1 and batch["labels"].shape[0] == batch["input_ids"].shape[0]
assert 1 <= len(batch["labels"]) <= BATCH_SIZE
assert all(t.dtype == torch.long for t in batch.values())
assert set(batch["labels"].tolist()).issubset({0, 1})
assert len(train_loader.dataset) == client0["n_train"]
assert len(val_loader.dataset) == client0["n_val"]
print("PASS TODO 2: shapes", {k: tuple(v.shape) for k, v in batch.items()})


PASS TODO 2: shapes {'input_ids': (32, 256), 'attention_mask': (32, 256), 'labels': (32,)}


## 3. TODO 3

You are **not required** to use exactly this architecture. Alternatives can include CNNs, recurrent layers, different pooling rules or a more capable model, **if you explain your choice and can run it within the available resources**. The model must:

1. accept `input_ids` and `attention_mask` tensors;
2. return **raw logits** of shape `[B, 2]` (do **not** apply softmax before `nn.CrossEntropyLoss`);
3. use the shared vocabulary and handle padding correctly;
4. be instantiated with the **same architecture** for every client.


**Your implementation.** Implement `build_model()`. You can define your own `nn.Module` class above or inside it. You control hidden dimensions, activation, dropout, pooling and depth. Leave a model-description note below.


In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pad_token_id):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_token_id)

        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=embed_dim,
            kernel_size=3,
            padding=1
        )
        self.relu = nn.ReLU()

        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2)
        )

    def forward(self, input_ids, attention_mask):

        x = self.embedding(input_ids)
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask

        x = x.transpose(1, 2)
        x = self.relu(self.conv(x))
        x = x.transpose(1, 2)

        masked_sum = (x * mask).sum(dim=1)
        real_token_counts = mask.sum(dim=1).clamp(min=1.0)
        pooled = masked_sum / real_token_counts

        return self.classifier(pooled)

def build_model():
    """Return a *new* nn.Module with forward(input_ids, attention_mask) -> [B, 2] logits.

    Available: VOCAB_SIZE, PAD_TOKEN_ID, EMBED_DIM, HIDDEN_DIM, DROPOUT.
    You may introduce other settings or use a different architecture.
    """
    # TODO 3 — YOUR MODEL ARCHITECTURE HERE
    # Suggested building blocks: nn.Embedding(..., padding_idx=PAD_TOKEN_ID),
    # nn.Linear(...), nn.ReLU(), nn.Dropout(...), masked pooling.
    # Make sure forward uses the attention_mask when required.

    return SentimentClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_DIM, DROPOUT, PAD_TOKEN_ID)


We use a 1D CNN: embedding -> conv1d + ReLU -> masked mean pooling -> linear + ReLU + dropout -> linear

In [ ]:
# CHECK 3 — do not edit
seed_everything(SEED)
model = build_model().to(DEVICE)
assert isinstance(model, nn.Module)
with torch.no_grad():
    logits = model(batch["input_ids"].to(DEVICE), batch["attention_mask"].to(DEVICE))
assert logits.shape == (len(batch["labels"]), 2), f"Expected [B, 2], got {tuple(logits.shape)}"
assert torch.isfinite(logits).all(), "Model returns invalid logits"
assert sum(p.numel() for p in model.parameters() if p.requires_grad) > 0
criterion = nn.CrossEntropyLoss()
initial_loss = criterion(logits, batch["labels"].to(DEVICE))
assert torch.isfinite(initial_loss)
print("PASS TODO 3: logits", tuple(logits.shape), "| parameters:",
      sum(p.numel() for p in model.parameters()), "| sample loss:", round(initial_loss.item(), 4))


PASS TODO 3: logits (32, 2) | parameters: 1970050 | sample loss: 0.6819


## 4. TODO 4

**Theory.** One training step follows:

`batch → forward pass → cross-entropy loss → backward pass (gradients) → optimizer step (updated weights)`

- **Loss** measures disagreement between predictions and correct labels.
- **Backpropagation** computes gradients; the **optimizer**, not `backward()`, updates the weights.
- **Validation** uses `model.eval()` and no gradient updates.
- Track *mean loss per example*, accuracy and macro-F1; accuracy can be deceptive on class-imbalanced clients.




In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Perform one complete training epoch; return mean training loss (float)."""
    # TODO 4A — YOUR CODE HERE
    # Hint: model.train(), zero_grad(), forward(), criterion(), backward(), step()
    # Weight losses by batch size before averaging across examples.

    model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    return total_loss / total_examples


def evaluate_model(model, loader, criterion, device):
    """No weight updates. Return {'loss':float, 'accuracy':float, 'macro_f1':float}.
    Accuracy and macro-F1 must be in [0,1].
    """
    # TODO 4B — YOUR CODE HERE
    # Hint: model.eval(), torch.no_grad(), argmax(dim=1), sklearn.metrics

    odel.eval()
    total_loss = 0.0
    total_examples = 0
    preds = []
    targets = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)

            total_loss += loss.item() * labels.size(0)
            total_examples += labels.size(0)

            preds.extend(logits.argmax(dim=1).cpu().tolist())
            targets.extend(labels.cpu().tolist())

    return {
        "loss": total_loss / total_examples,
        "accuracy": accuracy_score(targets, preds),
        "macro_f1": f1_score(targets, preds, average="macro"),
    }


In [ ]:
# This *orchestration* is provided; your TODO 4 functions implement the learning.
# Keep the first model's initialization: all later clients start from these same weights.
seed_everything(SEED)
reference_model = build_model().cpu()
COMMON_INITIAL_STATE = copy.deepcopy(reference_model.state_dict())

model0 = build_model().to(DEVICE)
model0.load_state_dict(COMMON_INITIAL_STATE)

# You may change the optimizer, but do so consistently when comparing clients.
optimizer0 = torch.optim.AdamW(model0.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
weights_before = [p.detach().cpu().clone() for p in model0.parameters() if p.requires_grad]

client0_history = []
for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model0, train_loader, optimizer0, criterion, DEVICE)
    val = evaluate_model(model0, val_loader, criterion, DEVICE)
    client0_history.append({"epoch": epoch, "train_loss": float(train_loss), **val})
    print(f"Epoch {epoch}: train_loss={train_loss:.4f} | val_loss={val['loss']:.4f} "
          f"| val_acc={val['accuracy']:.3f} | val_macro_f1={val['macro_f1']:.3f}")

# CHECK 4 — do not edit
weights_after = [p.detach().cpu() for p in model0.parameters() if p.requires_grad]
assert any(not torch.equal(a, b) for a, b in zip(weights_before, weights_after)), "Weights did not change"
assert len(client0_history) == EPOCHS
assert all(np.isfinite(h["train_loss"]) and np.isfinite(h["loss"]) for h in client0_history)
assert all(0 <= h["accuracy"] <= 1 and 0 <= h["macro_f1"] <= 1 for h in client0_history)
week1_ckpt = RESULTS_DIR / "week1_client_0.pt"
torch.save(model0.cpu().state_dict(), week1_ckpt)
model0.to(DEVICE)
print("PASS TODO 4: model trained and checkpoint saved to", week1_ckpt)


In [ ]:
# Plot the trajectory. Add more plots if they help you explain the learning behavior.
pd.DataFrame(client0_history).plot(x="epoch", y=["train_loss", "loss"], marker="o",
                                   title="Client 0 — training vs. validation loss")
plt.ylabel("Loss")
plt.grid(alpha=0.25)
plt.show()
